# Chapter 4.3: Parsing LLM Output into DataFrames

Goal: Parse simulated LLM JSON responses, handle malformed output, validate results, and build feature DataFrames ready for ML.

### Topics:
- Parsing JSON strings with `json.loads()`
- Cleaning malformed LLM responses (markdown wrapping, extra text)
- Validating extracted fields against expected schemas
- Batch processing with error handling
- Building DataFrames from extraction results
- One-hot encoding categorical features for ML

In [ ]:
import pandas as pd
import numpy as np
import json
import re

## Quick Recap

- **JSON**: A text format for structured data using key-value pairs, easily parsed by Python
- **`json.loads()`**: Converts a JSON string into a Python dictionary
- **Malformed response**: LLM output that isn't clean JSON (has markdown backticks, extra explanation, or formatting issues)
- **Validation**: Checking that parsed data has the expected fields, types, and values
- **Batch processing**: Running the same operation on many inputs, handling failures gracefully
- **One-hot encoding**: Converting categorical variables into binary columns (0/1) for ML models

## Data

We have 20 simulated LLM responses for product review sentiment extraction. In practice, these would come from an API call — here they're pre-generated strings with realistic formatting issues.

**Note on simulated LLM output:** In a real workflow, you'd call an API here. We use pre-generated responses so everyone gets the same results and no API keys are needed.

In [ ]:
# 20 simulated LLM responses — mix of clean, wrapped, and broken
llm_responses = [
    # Clean JSON (6 responses)
    '{"sentiment": "positive", "confidence": 0.95, "category": "coffee_maker", "mentions_price": true}',
    '{"sentiment": "negative", "confidence": 0.98, "category": "headphones", "mentions_price": true}',
    '{"sentiment": "mixed", "confidence": 0.65, "category": "charger", "mentions_price": true}',
    '{"sentiment": "negative", "confidence": 0.92, "category": "coffee_maker", "mentions_price": true}',
    '{"sentiment": "positive", "confidence": 0.97, "category": "headphones", "mentions_price": true}',
    '{"sentiment": "positive", "confidence": 0.88, "category": "charger", "mentions_price": true}',
    
    # Markdown-wrapped JSON (4 responses)
    '```json\n{"sentiment": "mixed", "confidence": 0.70, "category": "charger", "mentions_price": true}\n```',
    '```\n{"sentiment": "positive", "confidence": 0.91, "category": "coffee_maker", "mentions_price": true}\n```',
    '```json\n{"sentiment": "negative", "confidence": 0.85, "category": "headphones", "mentions_price": false}\n```',
    '```json\n{"sentiment": "positive", "confidence": 0.93, "category": "charger", "mentions_price": true}\n```',
    
    # Extra text before/after JSON (4 responses)
    'Here is my analysis:\n{"sentiment": "negative", "confidence": 0.89, "category": "coffee_maker", "mentions_price": false}',
    'Based on the review, I would classify it as:\n{"sentiment": "positive", "confidence": 0.94, "category": "headphones", "mentions_price": true}\nHope that helps!',
    '{"sentiment": "mixed", "confidence": 0.72, "category": "charger", "mentions_price": true}\n\nNote: The reviewer had mixed feelings about durability.',
    'Sure! {"sentiment": "positive", "confidence": 0.86, "category": "coffee_maker", "mentions_price": true}',
    
    # Missing fields (3 responses)
    '{"sentiment": "positive", "confidence": 0.90}',
    '{"sentiment": "negative", "category": "headphones", "mentions_price": true}',
    '{"confidence": 0.75, "category": "charger", "mentions_price": false}',
    
    # Wrong types (2 responses)
    '{"sentiment": "positive", "confidence": "high", "category": "coffee_maker", "mentions_price": true}',
    '{"sentiment": "POSITIVE", "confidence": 0.95, "category": "headphones", "mentions_price": "yes"}',
    
    # Completely broken (1 response)
    'I think the sentiment is positive. The customer really liked the product and would recommend it to others.'
]

In [ ]:
# Job posting extraction results (already parsed — a list of dicts)
job_extractions = [
    {"title": "Senior Data Scientist", "company": "TechCorp", "category": "data_science", "experience_level": "senior", "remote_policy": "remote", "skills": ["Python", "SQL", "machine learning"]},
    {"title": "Junior Web Developer", "company": "StartupXYZ", "category": "engineering", "experience_level": "entry", "remote_policy": "hybrid", "skills": ["React", "JavaScript", "CSS"]},
    {"title": "Marketing Manager", "company": "BigBrand Inc", "category": "marketing", "experience_level": "senior", "remote_policy": "on_site", "skills": ["SEO", "content strategy", "team leadership"]},
    {"title": "ML Engineer", "company": "DataFlow", "category": "data_science", "experience_level": "mid", "remote_policy": "remote", "skills": ["PyTorch", "MLOps", "cloud platforms"]},
    {"title": "Business Analyst", "company": "FinanceGroup", "category": "business", "experience_level": "entry", "remote_policy": "on_site", "skills": ["Excel", "SQL", "communication"]},
    {"title": "Principal Software Engineer", "company": "MegaTech", "category": "engineering", "experience_level": "principal", "remote_policy": "hybrid", "skills": ["system design", "distributed systems", "Go", "Rust"]},
    {"title": "Data Analyst Intern", "company": "HealthData", "category": "data_science", "experience_level": "entry", "remote_policy": "remote", "skills": ["Python", "Tableau", "statistics"]},
    {"title": "DevOps Lead", "company": "CloudNative Inc", "category": "engineering", "experience_level": "senior", "remote_policy": "remote", "skills": ["Kubernetes", "Terraform", "AWS", "CI/CD"]},
    {"title": "Product Manager", "company": "AppWorks", "category": "business", "experience_level": "mid", "remote_policy": "hybrid", "skills": ["agile", "roadmapping", "stakeholder management"]},
    {"title": "Frontend Developer", "company": "DesignFirst", "category": "engineering", "experience_level": "mid", "remote_policy": "remote", "skills": ["TypeScript", "React", "CSS", "Figma"]},
    {"title": "Data Engineer", "company": "PipelineIO", "category": "data_science", "experience_level": "mid", "remote_policy": "hybrid", "skills": ["Python", "SQL", "Spark", "Airflow"]},
    {"title": "UX Researcher", "company": "UserFirst", "category": "other", "experience_level": "mid", "remote_policy": "on_site", "skills": ["user interviews", "surveys", "A/B testing"]},
    {"title": "Security Engineer", "company": "SafeNet", "category": "engineering", "experience_level": "senior", "remote_policy": "remote", "skills": ["Python", "penetration testing", "cloud security"]},
    {"title": "Content Strategist", "company": "MediaPlus", "category": "marketing", "experience_level": "mid", "remote_policy": "hybrid", "skills": ["SEO", "copywriting", "analytics"]},
    {"title": "Backend Developer", "company": "APIStack", "category": "engineering", "experience_level": "senior", "remote_policy": "remote", "skills": ["Python", "Django", "PostgreSQL", "Redis"]}
]

## Practice

### 1. By hand — Parse clean and wrapped JSON

Start with `json.loads()` on a clean response. Then try it on a markdown-wrapped response — it will fail. Fix it by stripping the backticks.

In [ ]:
# Parse the first (clean) response
clean_response = llm_responses[0]
print("Raw string:", clean_response)

parsed = json.loads(clean_response)
print("Parsed:", parsed)
print("Sentiment:", parsed["sentiment"])

In [ ]:
# Now try a markdown-wrapped response — this will fail
wrapped_response = llm_responses[6]
print("Raw string:")
print(wrapped_response)
print()

# This will raise a JSONDecodeError — that's expected!
try:
    parsed = json.loads(wrapped_response)
except json.JSONDecodeError as e:
    print(f"Error: {e}")

In [ ]:
# Fix it: strip the markdown backticks and parse again
# Hint: remove lines that start with ``` 
cleaned = ...

parsed = json.loads(cleaned)
print("Parsed:", parsed)

### 2. By hand — Write `clean_llm_response()`

Write a function that handles all the common formatting issues:
1. Remove markdown code block markers (` ```json ` and ` ``` `)
2. Strip leading/trailing whitespace
3. Extract JSON from responses that have extra text before/after

Hint: JSON objects always start with `{` and end with `}`. You can find the first `{` and last `}` to extract the JSON portion.

In [ ]:
def clean_llm_response(response_str):
    """Clean a raw LLM response string so it can be parsed as JSON.
    
    Handles:
    - Markdown code block wrappers (```json ... ```)
    - Extra text before/after the JSON
    - Leading/trailing whitespace
    
    Returns the cleaned string (ready for json.loads), or the original
    string if no JSON object is found.
    """
    # Step 1: Strip whitespace
    ...
    
    # Step 2: Remove markdown code block markers
    ...
    
    # Step 3: Find the JSON object (first { to last })
    ...
    
    return ...

In [ ]:
# Test your function on different response types
test_cases = [
    ("Clean", llm_responses[0]),
    ("Markdown-wrapped", llm_responses[6]),
    ("Extra text before", llm_responses[10]),
    ("Extra text after", llm_responses[12]),
    ("No JSON at all", llm_responses[19]),
]

for label, response in test_cases:
    cleaned = clean_llm_response(response)
    try:
        parsed = json.loads(cleaned)
        print(f"{label}: OK — {parsed}")
    except json.JSONDecodeError:
        print(f"{label}: FAILED to parse — {cleaned[:60]}...")
    print()

### 3. By hand — Write `validate_extraction()`

Even when JSON parses correctly, the data might be wrong. Write a validation function that checks:
1. All required fields are present
2. Values are in the expected set (e.g., sentiment must be one of "positive", "negative", "mixed")
3. Types are correct (e.g., confidence must be a float)

Return a tuple of `(is_valid, error_messages)`.

In [ ]:
def validate_extraction(result, required_fields, valid_values=None):
    """Validate a parsed extraction result.
    
    Args:
        result: Dictionary from parsed JSON
        required_fields: List of field names that must be present
        valid_values: Dict mapping field names to allowed values,
                      e.g. {"sentiment": ["positive", "negative", "mixed"]}
    
    Returns:
        (is_valid, error_messages) tuple
    """
    errors = []
    
    # Check required fields
    ...
    
    # Check valid values
    ...
    
    # Check that confidence is a number between 0 and 1 (if present)
    ...
    
    return (len(errors) == 0, errors)

In [ ]:
# Test your validation function
required = ["sentiment", "confidence", "category", "mentions_price"]
valid = {
    "sentiment": ["positive", "negative", "mixed"],
    "category": ["coffee_maker", "headphones", "charger"]
}

# Test on clean response
good_result = json.loads(llm_responses[0])
is_valid, errs = validate_extraction(good_result, required, valid)
print(f"Clean response: valid={is_valid}, errors={errs}")

# Test on response with missing fields
missing_result = json.loads(llm_responses[14])
is_valid, errs = validate_extraction(missing_result, required, valid)
print(f"Missing fields: valid={is_valid}, errors={errs}")

# Test on response with wrong types
wrong_type = json.loads(llm_responses[17])
is_valid, errs = validate_extraction(wrong_type, required, valid)
print(f"Wrong types: valid={is_valid}, errors={errs}")

### 4. Use AI — Batch processing pipeline

Use your AI assistant to write a `process_batch()` function that:
1. Takes a list of raw LLM response strings
2. Cleans each one with `clean_llm_response()`
3. Tries to parse as JSON
4. Validates with `validate_extraction()`
5. Returns two lists: `valid_results` and `failed_results` (with error info)

Run it on all 20 responses and report the success rate.

In [ ]:
# Use AI to implement this function
def process_batch(responses, required_fields, valid_values=None):
    """Process a batch of LLM responses: clean, parse, validate.
    
    Returns:
        (valid_results, failed_results) where each is a list of dicts
    """
    ...

# Run on all 20 responses
required = ["sentiment", "confidence", "category", "mentions_price"]
valid = {
    "sentiment": ["positive", "negative", "mixed"],
    "category": ["coffee_maker", "headphones", "charger"]
}

valid_results, failed_results = process_batch(llm_responses, required, valid)

print(f"Total: {len(llm_responses)}")
print(f"Valid: {len(valid_results)}")
print(f"Failed: {len(failed_results)}")
print(f"Success rate: {len(valid_results) / len(llm_responses):.0%}")
print()
print("Failed responses:")
for f in failed_results:
    print(f"  Index {f['index']}: {f['error']}")

### 5. Use AI — Build a feature DataFrame

Use your AI assistant to convert the valid results into a pandas DataFrame. Then add a `review_length` column (you can use index as a proxy for which review it was, and assign random lengths for this exercise).

In [ ]:
# Use AI to convert valid_results to a DataFrame
...

# Add a review_length column (simulated)
np.random.seed(42)
...

# Display the DataFrame
...

### 6. By hand — Work with job posting extractions

The `job_extractions` list contains pre-parsed extraction results for 15 job postings. Convert it to a DataFrame and add two new columns:
- `num_skills`: the number of skills listed
- `requires_python`: boolean — does the skills list contain "Python"?

In [ ]:
# Convert to DataFrame
jobs_df = pd.DataFrame(job_extractions)
jobs_df.head()

In [ ]:
# Add num_skills column (count items in the skills list)
jobs_df["num_skills"] = ...

# Add requires_python column (True if "Python" is in skills)
jobs_df["requires_python"] = ...

jobs_df[["title", "skills", "num_skills", "requires_python"]].head(10)

### 7. Use AI — One-hot encode categorical columns

Use your AI assistant to one-hot encode the `category`, `experience_level`, and `remote_policy` columns using `pd.get_dummies()`. Drop the original columns and the `skills` list column (since ML models can't use lists directly).

In [ ]:
# Use AI to one-hot encode and prepare the DataFrame for ML
...

# Display the result
...

### 8. Interpretation — Handling failures at scale

Imagine you're running this pipeline on 10,000 product reviews and 15% of responses fail parsing or validation. What would you do with the failed extractions?

**Your analysis:** Consider these options and discuss the tradeoffs of each:
1. Drop the failed rows entirely
2. Retry with a different prompt
3. Fill in default/missing values
4. Use a fallback method (regex, keywords) for failed rows

(Write your answer here)

## Discussion

If an LLM gives you perfectly formatted JSON every time, does that mean the extracted information is correct? What's the difference between parsing success and extraction accuracy?

(Discuss with a neighbor)